# YOLO26x Vast.ai Training

Run this notebook with the `Python (grad)` kernel. All paths are local to the current working directory, normally `/root` on this Vast instance.


In [1]:

import os
import subprocess
import sys
from pathlib import Path

WORK_DIR = Path.cwd().resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

# This Vast instance has one visible GPU. Keep all CUDA work on GPU 0.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("MKL_NUM_THREADS", "8")

print("Working directory:", WORK_DIR)
print("Kernel executable:", sys.executable)

if "grad" not in Path(sys.executable).parts:
    print("WARNING: switch the notebook kernel to `Python (grad)` before training.")

try:
    import torch
    import ultralytics
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Missing packages in this kernel. Use the `Python (grad)` kernel, or run: "
        "/venv/grad/bin/python -m pip install -r requirements.txt"
    ) from exc

# Faster matmul defaults for recent NVIDIA GPUs.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout or result.stderr)

print("Python          :", sys.version.split()[0])
print("PyTorch version :", torch.__version__)
print("Ultralytics     :", ultralytics.__version__)
print("CUDA available  :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Make sure the Vast instance was started with a GPU image.")

GPU_COUNT = torch.cuda.device_count()
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU count       :", GPU_COUNT)
print("GPU             :", GPU_NAME)
print("VRAM            :", round(GPU_VRAM_GB, 2), "GiB")


Working directory: /root
Kernel executable: /venv/grad/bin/python
Wed Apr 29 10:04:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 4000 Blac...    On  |   00000000:C2:00.0 Off |                  Off |
| 30%   26C    P8              2W /  145W |   19097MiB /  24467MiB |      0%      Default |
|                                         |                        |      

## Paths And Dataset YAML

This cell finds or downloads the SYN-COCO Roboflow dataset and always writes the fixed YAML used by training.


In [2]:

from pathlib import Path
import os
import yaml

WORK_DIR = Path.cwd().resolve()
os.chdir(WORK_DIR)

try:
    from dotenv import load_dotenv
    load_dotenv(WORK_DIR / "pipeline" / ".env")
except Exception:
    pass

ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY", "").strip()
ROBOFLOW_WORKSPACE = "munisdrafts"
ROBOFLOW_PROJECT = "syn-coco-qdonc-hopnq"
ROBOFLOW_VERSION = 2
ROBOFLOW_FORMAT = "yolo26"

DATASET_DIR = WORK_DIR / "syn-coco-2"
ORIGINAL_YAML = DATASET_DIR / "data.yaml"
FIXED_YAML = DATASET_DIR / "data_fixed.yaml"


def find_dataset_dir() -> Path | None:
    preferred = [DATASET_DIR, WORK_DIR / "Syn-Coco-2", WORK_DIR / "syn-coco"]
    for path in preferred:
        if (path / "data.yaml").exists():
            return path.resolve()

    ignored = {".cache", ".conda", ".config", ".codex", ".local", ".vscode-server", "YOLO26", "coco_val2017_yolo_REAL"}
    for child in sorted(WORK_DIR.iterdir()):
        if not child.is_dir() or child.name in ignored or child.name.startswith("."):
            continue
        if (child / "data.yaml").exists():
            return child.resolve()
        for grandchild in sorted(child.iterdir()):
            if grandchild.is_dir() and (grandchild / "data.yaml").exists():
                return grandchild.resolve()
    return None


def download_syn_coco_dataset() -> Path:
    from roboflow import Roboflow

    if not ROBOFLOW_API_KEY:
        raise RuntimeError("ROBOFLOW_API_KEY env var is required to download from Roboflow.")
    print("SYN-COCO data.yaml was not found; downloading Roboflow dataset into:", DATASET_DIR)
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    dataset = version.download(ROBOFLOW_FORMAT, location=str(DATASET_DIR), overwrite=False)
    return Path(dataset.location).resolve()


def ensure_syn_coco_dataset() -> Path:
    global DATASET_DIR, ORIGINAL_YAML, FIXED_YAML

    found = find_dataset_dir()
    if found is None:
        found = download_syn_coco_dataset()

    DATASET_DIR = found
    ORIGINAL_YAML = DATASET_DIR / "data.yaml"
    FIXED_YAML = DATASET_DIR / "data_fixed.yaml"

    if not ORIGINAL_YAML.exists():
        raise FileNotFoundError(
            f"Dataset data.yaml is missing at {ORIGINAL_YAML}. "
            "Run the Roboflow download cell again or place the dataset in the notebook directory."
        )
    return DATASET_DIR


def get_images_dir(split_name: str) -> str:
    split_folder = DATASET_DIR / split_name
    if (split_folder / "images").exists():
        return f"{split_name}/images"
    if split_folder.exists():
        return split_name
    raise FileNotFoundError(f"Could not find split folder: {split_folder}")


def count_images(relative_dir: str) -> int:
    image_dir = DATASET_DIR / relative_dir
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sum(1 for p in image_dir.rglob("*") if p.suffix.lower() in exts)


def count_labels(split_name: str) -> int:
    label_dir = DATASET_DIR / split_name / "labels"
    if not label_dir.exists():
        return 0
    return len(list(label_dir.rglob("*.txt")))


def load_yolo_names(yaml_path: Path):
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    names = data["names"]
    if isinstance(names, dict):
        names = [names[i] if i in names else names[str(i)] for i in range(len(names))]
    elif not isinstance(names, list):
        raise TypeError("Unsupported names format in data.yaml")

    return [str(x).strip() for x in names]


def ensure_training_yaml() -> Path:
    global DATASET_DIR, ORIGINAL_YAML, FIXED_YAML

    ensure_syn_coco_dataset()

    with open(ORIGINAL_YAML, "r") as f:
        data = yaml.safe_load(f)

    valid_split_name = "valid" if (DATASET_DIR / "valid").exists() else "val"
    train_dir = get_images_dir("train")
    val_dir = get_images_dir(valid_split_name)

    data["path"] = str(DATASET_DIR)
    data["train"] = train_dir
    data["val"] = val_dir

    test_folder = DATASET_DIR / "test"
    if test_folder.exists():
        try:
            data["test"] = get_images_dir("test")
        except FileNotFoundError:
            data.pop("test", None)
    else:
        data.pop("test", None)

    with open(FIXED_YAML, "w") as f:
        yaml.safe_dump(data, f, sort_keys=False)

    train_names = load_yolo_names(FIXED_YAML)
    print("Dataset directory:", DATASET_DIR)
    print("Fixed YAML saved to:", FIXED_YAML)
    print("Number of classes:", len(train_names))
    print("Train images:", count_images(train_dir))
    print("Valid images:", count_images(val_dir))
    print("Train labels:", count_labels("train"))
    print("Valid labels:", count_labels(valid_split_name))
    return FIXED_YAML


FIXED_YAML = ensure_training_yaml()
print("\nFixed YAML contents:")
print("-" * 70)
print(FIXED_YAML.read_text())


Dataset directory: /root/syn-coco-2
Fixed YAML saved to: /root/syn-coco-2/data_fixed.yaml
Number of classes: 81
Train images: 12809
Valid images: 3202
Train labels: 12809
Valid labels: 3202

Fixed YAML contents:
----------------------------------------------------------------------
train: train/images
val: valid/images
nc: 81
names:
- airplane
- apple
- backpack
- banana
- baseball bat
- baseball glove
- bear
- bed
- bench
- bicycle
- bird
- boat
- book
- bottle
- bowl
- broccoli
- bus
- cake
- car
- carrot
- cat
- cell phone
- chair
- clock
- couch
- cow
- cup
- dining table
- dog
- donut
- elephant
- fire hydrant
- fork
- frisbee
- giraffe
- hair drier
- handbag
- horse
- hot dog
- keyboard
- kite
- knife
- laptop
- microwave
- motorcycle
- mouse
- orange
- oven
- parking meter
- person
- pizza
- potted plant
- refrigerator
- remote
- sandwich
- scissors
- sheep
- sink
- skateboard
- skis
- snowboard
- spoon
- sports ball
- stop sign
- suitcase
- surfboard
- teddy bear
- tennis racke

## COCO Val2017 Download

Needed only for the later real COCO evaluation cells.


In [3]:

from pathlib import Path
import zipfile
import urllib.request

WORK_DIR = Path.cwd().resolve()

VAL_ZIP = WORK_DIR / "val2017.zip"
ANN_ZIP = WORK_DIR / "annotations_trainval2017.zip"

VAL_DIR = WORK_DIR / "val2017"
ANN_DIR = WORK_DIR / "annotations"
INSTANCES_VAL_JSON = ANN_DIR / "instances_val2017.json"

VAL_URL = "http://images.cocodataset.org/zips/val2017.zip"
ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"


def download_file(url, output_path):
    if output_path.exists():
        print(f"Already downloaded: {output_path}")
        return
    print(f"Downloading: {url}")
    urllib.request.urlretrieve(url, output_path)
    print(f"Saved to: {output_path}")


def unzip_file(zip_path, extract_to):
    print(f"Unzipping: {zip_path} -> {extract_to}")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_to)
    print("Done.")


if not VAL_DIR.exists() or len(list(VAL_DIR.glob("*.jpg"))) != 5000:
    download_file(VAL_URL, VAL_ZIP)
    unzip_file(VAL_ZIP, WORK_DIR)
else:
    print("COCO val2017 images already exist.")

if not INSTANCES_VAL_JSON.exists():
    download_file(ANN_URL, ANN_ZIP)
    unzip_file(ANN_ZIP, WORK_DIR)
else:
    print("COCO annotations already exist.")

print("\nFinal check:")
print(f"{VAL_DIR} exists:", VAL_DIR.exists())
print("Number of val2017 images:", len(list(VAL_DIR.glob("*.jpg"))))
print(f"{INSTANCES_VAL_JSON} exists:", INSTANCES_VAL_JSON.exists())


COCO val2017 images already exist.
COCO annotations already exist.

Final check:
/root/val2017 exists: True
Number of val2017 images: 5000
/root/annotations/instances_val2017.json exists: True


## Training Config


In [4]:

from pathlib import Path
import os
import json
import yaml
import shutil
from collections import defaultdict

import torch
from ultralytics import YOLO
from tqdm import tqdm

os.environ["WANDB_DISABLED"] = "true"
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

WORK_DIR = Path.cwd().resolve()
os.chdir(WORK_DIR)

# Make this cell safe even if the YAML cell was skipped.
FIXED_YAML = ensure_training_yaml()

RAW_COCO_IMAGES = WORK_DIR / "val2017"
COCO_ANN_JSON = WORK_DIR / "annotations" / "instances_val2017.json"

PROJECT_DIR = WORK_DIR / "YOLO26"
RUN_NAME = "YOLO26x-best-hyp"

BASE_MODEL_NAME = "yolo26x-objv1-150.pt"
REQUESTED_MODEL_NAME = "yolo26x-objv1-150.pt"
BASE_MODEL_PATH = WORK_DIR / BASE_MODEL_NAME
REQUESTED_MODEL_PATH = WORK_DIR / REQUESTED_MODEL_NAME
IMG_SIZE = 640
EPOCHS = 1

# Your pasted source run used two GPUs (`0,1`) and batch 128.
# This Vast instance has one RTX PRO 4000 Blackwell with about 24 GiB VRAM.
DEVICE = 0
WORKERS = 8
REQUESTED_BATCH = 64
BATCH_FALLBACKS = [64]
VAL_BATCH_FALLBACKS = [64]

RUN_TRAINING = True

TRAIN_ARGS = {
    "task": "detect",
    "mode": "train",
    "model": REQUESTED_MODEL_NAME,
    "data": str(FIXED_YAML),
    "epochs": 40,
    "time": None,
    "patience": 100,
    "batch": REQUESTED_BATCH,
    "imgsz": 640,
    "save": True,
    "save_period": -1,
    "cache": False,
    "device": DEVICE,
    "workers": WORKERS,
    "project": str(PROJECT_DIR),
    "name": RUN_NAME,
    "exist_ok": False,
    "pretrained": True,
    "optimizer": "MuSGD",
    "verbose": True,
    "seed": 0,
    "deterministic": True,
    "single_cls": False,
    "rect": False,
    "cos_lr": False,
    "close_mosaic": 10,
    "resume": False,
    "amp": True,
    "fraction": 1.0,
    "profile": False,
    "freeze": None,
    "multi_scale": 0.0,
    "compile": False,
    "overlap_mask": True,
    "mask_ratio": 4,
    "dropout": 0.0,
    "val": True,
    "split": "val",
    "save_json": True,
    "conf": None,
    "iou": 0.7,
    "max_det": 300,
    "half": False,
    "dnn": False,
    "plots": True,
    "source": None,
    "vid_stride": 1,
    "stream_buffer": False,
    "visualize": False,
    "augment": False,
    "agnostic_nms": False,
    "classes": None,
    "retina_masks": False,
    "embed": None,
    "show": False,
    "save_frames": False,
    "save_txt": False,
    "save_conf": False,
    "save_crop": False,
    "show_labels": True,
    "show_conf": True,
    "show_boxes": True,
    "line_width": None,
    "format": "torchscript",
    "keras": False,
    "optimize": False,
    "int8": False,
    "dynamic": False,
    "simplify": True,
    "opset": None,
    "workspace": None,
    "nms": False,
    "lr0": 0.00038,
    "lrf": 0.88219,
    "momentum": 0.94751,
    "weight_decay": 0.00027,
    "warmup_epochs": 0.98745,
    "warmup_momentum": 0.54064,
    "warmup_bias_lr": 0.05684,
    "o2m": 0.70518,
    "muon_w": 0.4355,
    "sgd_w": 0.47908,
    "cls_w": 3.48357,
    "stride_ratio": 1.0,
    "detach_epoch": 10,
    "topk": 5,
    "end2end": True,
    "box": 9.83241,
    "cls": 0.64896,
    "dfl": 0.95824,
    "pose": 12.0,
    "kobj": 1.0,
    "nbs": 64,
    "hsv_h": 0.01315,
    "hsv_s": 0.35348,
    "hsv_v": 0.19383,
    "degrees": 0.00012,
    "translate": 0.27484,
    "scale": 0.95,
    "shear": 0.00136,
    "perspective": 0.00074,
    "flipud": 0.00653,
    "fliplr": 0.30393,
    "bgr": 0.0,
    "mosaic": 0.99182,
    "mixup": 0.42713,
    "cutmix": 0.00082,
    "copy_paste": 0.40413,
    "copy_paste_mode": "flip",
    "auto_augment": "randaugment",
    "erasing": 0.4,
    "cfg": None,
    "tracker": "botsort.yaml",
}

# Bookkeeping/custom keys from the source run. Keep them documented above,
# but do not pass them directly to public Ultralytics train().
PUBLIC_ULTRALYTICS_UNSUPPORTED_ARGS = {
    "task",
    "mode",
    "model",
    "detach_epoch",
    "topk",
    "sgd_w",
    "cls_w",
    "stride_ratio",
    "o2m",
    "muon_w",
}

COCO_EVAL_ROOT = WORK_DIR / "coco_val2017_yolo_REAL"
COCO_EVAL_IMAGES_DIR = COCO_EVAL_ROOT / "images" / "val2017"
COCO_EVAL_LABELS_DIR = COCO_EVAL_ROOT / "labels" / "val2017"
COCO_EVAL_YAML = COCO_EVAL_ROOT / "coco_val2017_REAL.yaml"

print("Working directory:", WORK_DIR)
print("Dataset:", DATASET_DIR)
print("Training YAML:", FIXED_YAML, "exists:", FIXED_YAML.exists())
print("COCO images:", RAW_COCO_IMAGES)
print("COCO annotation:", COCO_ANN_JSON)
print("Requested checkpoint:", REQUESTED_MODEL_PATH)
print("Fallback checkpoint:", BASE_MODEL_PATH)
print("Project:", PROJECT_DIR)
print("Run name:", RUN_NAME)
print("GPU device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Train batch request/fallbacks:", BATCH_FALLBACKS)


Dataset directory: /root/syn-coco-2
Fixed YAML saved to: /root/syn-coco-2/data_fixed.yaml
Number of classes: 81
Train images: 12809
Valid images: 3202
Train labels: 12809
Valid labels: 3202
Working directory: /root
Dataset: /root/syn-coco-2
Training YAML: /root/syn-coco-2/data_fixed.yaml exists: True
COCO images: /root/val2017
COCO annotation: /root/annotations/instances_val2017.json
Requested checkpoint: /root/yolo26x-objv1-150.pt
Fallback checkpoint: /root/yolo26x-objv1-150.pt
Project: /root/YOLO26
Run name: YOLO26x-best-hyp
GPU device: 0 NVIDIA RTX PRO 4000 Blackwell
Train batch request/fallbacks: [64]


## Inspect Starting Weights


In [5]:

from ultralytics import YOLO

if REQUESTED_MODEL_PATH.exists():
    inspect_model_path = REQUESTED_MODEL_PATH
else:
    inspect_model_path = BASE_MODEL_PATH if BASE_MODEL_PATH.exists() else BASE_MODEL_NAME

model = YOLO(str(inspect_model_path))
print("Inspecting weights:", inspect_model_path)
print(model.ckpt["train_args"])


Inspecting weights: /root/yolo26x-objv1-150.pt
{'task': 'detect', 'mode': 'train', 'model': 'yolo26x.yaml', 'data': 'runs/data/Objects365v1.yaml', 'epochs': 150, 'time': None, 'patience': 100, 'batch': 128, 'imgsz': 640, 'save': True, 'save_period': -1, 'cache': False, 'device': '5,6', 'workers': 8, 'project': 'exp13-e2e', 'name': 'x-muon-0.50-sgd-0.60-150', 'exist_ok': False, 'pretrained': True, 'optimizer': 'MuSGD', 'verbose': True, 'seed': 0, 'deterministic': True, 'single_cls': False, 'rect': False, 'cos_lr': False, 'close_mosaic': 8, 'resume': False, 'amp': True, 'fraction': 1.0, 'profile': False, 'freeze': None, 'multi_scale': 0.0, 'compile': True, 'overlap_mask': True, 'mask_ratio': 4, 'dropout': 0.0, 'val': True, 'split': 'val', 'save_json': True, 'conf': None, 'iou': 0.7, 'max_det': 300, 'half': False, 'dnn': False, 'plots': True, 'source': None, 'vid_stride': 1, 'stream_buffer': False, 'visualize': False, 'augment': False, 'agnostic_nms': False, 'classes': None, 'retina_masks

## Train YOLO26x


In [6]:

def resolve_starting_weights() -> Path:
    if REQUESTED_MODEL_PATH.exists():
        return REQUESTED_MODEL_PATH

    if not BASE_MODEL_PATH.exists():
        print(f"Downloading fallback checkpoint: {BASE_MODEL_NAME}")
        YOLO(BASE_MODEL_NAME)

    return BASE_MODEL_PATH


# Critical path fix: always create/refresh the local YAML before training.
FIXED_YAML = ensure_training_yaml()
assert FIXED_YAML.exists(), f"Missing training YAML after path fix: {FIXED_YAML}"
TRAIN_ARGS["data"] = str(FIXED_YAML)

STARTING_WEIGHTS = resolve_starting_weights()
print("Starting weights:", STARTING_WEIGHTS)

if RUN_TRAINING:
    skipped_train_args = {
        key: TRAIN_ARGS[key]
        for key in sorted(PUBLIC_ULTRALYTICS_UNSUPPORTED_ARGS)
        if key in TRAIN_ARGS
    }
    train_overrides = {
        key: value
        for key, value in TRAIN_ARGS.items()
        if key not in PUBLIC_ULTRALYTICS_UNSUPPORTED_ARGS
    }
    train_overrides.update(
        data=str(FIXED_YAML),
        device=DEVICE,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
        workers=WORKERS,
    )

    if skipped_train_args:
        print("Filtering bookkeeping/custom args before model.train():")
        for key, value in skipped_train_args.items():
            print(f"  {key}: {value}")

    last_error = None
    for batch in BATCH_FALLBACKS:
        train_overrides["batch"] = batch
        TRAIN_ARGS["batch"] = batch
        try:
            torch.cuda.empty_cache()
            model = YOLO(str(STARTING_WEIGHTS))
            print(f"\nStarting training with batch={batch} on device={DEVICE}")
            train_results = model.train(**train_overrides)
            break
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            last_error = exc
            print(f"CUDA out of memory at batch={batch}; retrying smaller batch.")
            del model
            torch.cuda.empty_cache()
    else:
        raise RuntimeError("All batch sizes failed with CUDA out-of-memory.") from last_error

    RUN_DIR = Path(train_results.save_dir)
    BEST_PT = RUN_DIR / "weights" / "best.pt"
    LAST_PT = RUN_DIR / "weights" / "last.pt"
    TRAINED_WEIGHTS = BEST_PT if BEST_PT.exists() else LAST_PT
else:
    RUN_DIR = PROJECT_DIR / RUN_NAME
    BEST_PT = RUN_DIR / "weights" / "best.pt"
    LAST_PT = RUN_DIR / "weights" / "last.pt"
    if BEST_PT.exists():
        TRAINED_WEIGHTS = BEST_PT
    elif LAST_PT.exists():
        TRAINED_WEIGHTS = LAST_PT
    else:
        raise FileNotFoundError(
            f"No trained weights found for {RUN_NAME}. Set RUN_TRAINING=True and run this cell."
        )

print("Using trained weights:", TRAINED_WEIGHTS)
assert Path(TRAINED_WEIGHTS).exists(), f"Missing trained weights: {TRAINED_WEIGHTS}"


Dataset directory: /root/syn-coco-2
Fixed YAML saved to: /root/syn-coco-2/data_fixed.yaml
Number of classes: 81
Train images: 12809
Valid images: 3202
Train labels: 12809
Valid labels: 3202
Starting weights: /root/yolo26x-objv1-150.pt
Filtering bookkeeping/custom args before model.train():
  cls_w: 3.48357
  detach_epoch: 10
  mode: train
  model: yolo26x-objv1-150.pt
  muon_w: 0.4355
  o2m: 0.70518
  sgd_w: 0.47908
  stride_ratio: 1.0
  task: detect
  topk: 5

Starting training with batch=64 on device=0
New https://pypi.org/project/ultralytics/8.4.43 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu130 CUDA:0 (NVIDIA RTX PRO 4000 Blackwell, 23988MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=9.83241, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.64896, cls_pw=0.0, compile=False, conf=None, copy_paste=0.40413, copy_paste_mode=flip, cos_lr=F

## Validate On SYN-COCO


In [7]:

def val_with_batch_fallback(model, *, data, name):
    last_error = None
    for batch in VAL_BATCH_FALLBACKS:
        try:
            torch.cuda.empty_cache()
            print(f"\nStarting validation with batch={batch}")
            return model.val(
                data=str(data),
                split="val",
                imgsz=IMG_SIZE,
                batch=batch,
                device=DEVICE,
                workers=WORKERS,
                half=True,
                plots=True,
                save_json=True,
                project=str(PROJECT_DIR),
                name=name,
                exist_ok=True,
                verbose=True,
            )
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            last_error = exc
            print(f"CUDA out of memory at validation batch={batch}; retrying smaller batch.")
            torch.cuda.empty_cache()
    raise RuntimeError("All validation batch sizes failed with CUDA out-of-memory.") from last_error


FIXED_YAML = ensure_training_yaml()
trained_model = YOLO(str(TRAINED_WEIGHTS))

syn_metrics = val_with_batch_fallback(
    trained_model,
    data=FIXED_YAML,
    name=f"{RUN_NAME}_syn_valid_eval",
)

print("\n==============================")
print("SYN-COCO valid results")
print("==============================")
print("mAP50-95:", syn_metrics.box.map)
print("mAP50:", syn_metrics.box.map50)
print("mAP75:", syn_metrics.box.map75)
print("Results saved to:", syn_metrics.save_dir)


Dataset directory: /root/syn-coco-2
Fixed YAML saved to: /root/syn-coco-2/data_fixed.yaml
Number of classes: 81
Train images: 12809
Valid images: 3202
Train labels: 12809
Valid labels: 3202

Starting validation with batch=64
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu130 CUDA:0 (NVIDIA RTX PRO 4000 Blackwell, 23988MiB)
YOLO26x summary (fused): 190 layers, 55,727,103 parameters, 0 gradients, 193.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1684.6±323.2 MB/s, size: 78.7 KB)
val: Scanning /root/syn-coco-2/valid/labels.cache... 3202 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3202/3202 706.9Mit/s 0.0s
val: /root/syn-coco-2/valid/images/apple_026_png.rf.0af89ec6d1cc8960e9e587d14e57930c.jpg: 1 duplicate labels removed
val: /root/syn-coco-2/valid/images/backpack_060_png.rf.f43b55169b29bcd86cce4ff2dac82318.jpg: 1 duplicate labels removed
val: /root/syn-coco-2/valid/images/backpack_073_png.rf.c0ecc8488443acd0486815eaf57ab92d.jpg: 2 duplicate labels removed
val: /ro

## Prepare COCO Val2017 Evaluation Folders


In [8]:

assert RAW_COCO_IMAGES.exists(), f"Missing COCO val image folder: {RAW_COCO_IMAGES}. Run the COCO download cell."
assert COCO_ANN_JSON.exists(), f"Missing COCO annotation JSON: {COCO_ANN_JSON}. Run the COCO download cell."

COCO_EVAL_IMAGES_DIR.parent.mkdir(parents=True, exist_ok=True)
COCO_EVAL_LABELS_DIR.mkdir(parents=True, exist_ok=True)

if COCO_EVAL_IMAGES_DIR.is_symlink():
    COCO_EVAL_IMAGES_DIR.unlink()

COCO_EVAL_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

image_files = sorted(RAW_COCO_IMAGES.glob("*.jpg"))
print("Found raw COCO images:", len(image_files))
assert len(image_files) > 0, f"No .jpg files found in {RAW_COCO_IMAGES}"

for src_path in image_files:
    dst = COCO_EVAL_IMAGES_DIR / src_path.name
    if dst.exists():
        continue
    try:
        os.link(src_path, dst)
    except OSError:
        shutil.copy2(src_path, dst)

print("Images inside corrected folder:", len(list(COCO_EVAL_IMAGES_DIR.glob("*.jpg"))))
print("Correct image folder:", COCO_EVAL_IMAGES_DIR)


Found raw COCO images: 5000
Images inside corrected folder: 5000
Correct image folder: /root/coco_val2017_yolo_REAL/images/val2017


## Convert COCO Annotations To YOLO Labels


In [9]:

for p in COCO_EVAL_LABELS_DIR.glob("*.txt"):
    p.unlink()

train_names = load_yolo_names(FIXED_YAML)
train_name_to_idx = {name.lower(): i for i, name in enumerate(train_names)}

with open(COCO_ANN_JSON, "r") as f:
    coco = json.load(f)

coco_id_to_name = {cat["id"]: cat["name"].strip() for cat in coco["categories"]}

missing = []
for cat in coco["categories"]:
    if cat["name"].strip().lower() not in train_name_to_idx:
        missing.append(cat["name"])

if missing:
    print("Missing class names from your data.yaml:")
    print(missing)
    raise RuntimeError("Your SYN-COCO class names do not exactly match COCO names.")

print("Class-name check passed.")
print("Your data.yaml classes:", len(train_names))
print("COCO categories:", len(coco["categories"]))

image_id_to_info = {img["id"]: img for img in coco["images"]}

for img in coco["images"]:
    label_file = COCO_EVAL_LABELS_DIR / (Path(img["file_name"]).stem + ".txt")
    label_file.write_text("")

labels_by_image = defaultdict(list)
converted = 0
skipped_crowd = 0
skipped_bad_box = 0

for ann in coco["annotations"]:
    if ann.get("iscrowd", 0) == 1:
        skipped_crowd += 1
        continue

    img = image_id_to_info[ann["image_id"]]
    img_w = img["width"]
    img_h = img["height"]
    file_name = img["file_name"]

    class_name = coco_id_to_name[ann["category_id"]].lower()
    class_idx = train_name_to_idx[class_name]

    x, y, w, h = ann["bbox"]
    x1 = max(0.0, x)
    y1 = max(0.0, y)
    x2 = min(float(img_w), x + w)
    y2 = min(float(img_h), y + h)

    bw = x2 - x1
    bh = y2 - y1
    if bw <= 1 or bh <= 1:
        skipped_bad_box += 1
        continue

    x_center = (x1 + bw / 2) / img_w
    y_center = (y1 + bh / 2) / img_h
    bw_norm = bw / img_w
    bh_norm = bh / img_h

    labels_by_image[file_name].append(
        f"{class_idx} {x_center:.6f} {y_center:.6f} {bw_norm:.6f} {bh_norm:.6f}"
    )
    converted += 1

for file_name, lines in labels_by_image.items():
    label_file = COCO_EVAL_LABELS_DIR / (Path(file_name).stem + ".txt")
    label_file.write_text("\n".join(lines) + "\n")

print("Converted annotations:", converted)
print("Skipped crowd annotations:", skipped_crowd)
print("Skipped bad boxes:", skipped_bad_box)
print("Label files:", len(list(COCO_EVAL_LABELS_DIR.glob("*.txt"))))

assert converted > 0, "No annotations converted. COCO evaluation would be invalid."


Class-name check passed.
Your data.yaml classes: 81
COCO categories: 80
Converted annotations: 36334
Skipped crowd annotations: 446
Skipped bad boxes: 1
Label files: 5000


## Create COCO Eval YAML


In [10]:

eval_yaml = {
    "path": str(COCO_EVAL_ROOT),
    "train": "images/val2017",
    "val": "images/val2017",
    "names": {i: name for i, name in enumerate(train_names)},
}

with open(COCO_EVAL_YAML, "w") as f:
    yaml.safe_dump(eval_yaml, f, sort_keys=False)

print("Saved corrected COCO eval YAML:", COCO_EVAL_YAML)
print(COCO_EVAL_YAML.read_text())

cache_candidates = [
    RAW_COCO_IMAGES.with_suffix(".cache"),
    COCO_EVAL_ROOT / "labels" / "val2017.cache",
    COCO_EVAL_ROOT / "images" / "val2017.cache",
    COCO_EVAL_IMAGES_DIR.with_suffix(".cache"),
    COCO_EVAL_LABELS_DIR.with_suffix(".cache"),
]

for cache_path in cache_candidates:
    if cache_path.exists():
        cache_path.unlink()
        print("Deleted cache:", cache_path)


Saved corrected COCO eval YAML: /root/coco_val2017_yolo_REAL/coco_val2017_REAL.yaml
path: /root/coco_val2017_yolo_REAL
train: images/val2017
val: images/val2017
names:
  0: airplane
  1: apple
  2: backpack
  3: banana
  4: baseball bat
  5: baseball glove
  6: bear
  7: bed
  8: bench
  9: bicycle
  10: bird
  11: boat
  12: book
  13: bottle
  14: bowl
  15: broccoli
  16: bus
  17: cake
  18: car
  19: carrot
  20: cat
  21: cell phone
  22: chair
  23: clock
  24: couch
  25: cow
  26: cup
  27: dining table
  28: dog
  29: donut
  30: elephant
  31: fire hydrant
  32: fork
  33: frisbee
  34: giraffe
  35: hair drier
  36: handbag
  37: horse
  38: hot dog
  39: keyboard
  40: kite
  41: knife
  42: laptop
  43: microwave
  44: motorcycle
  45: mouse
  46: orange
  47: oven
  48: parking meter
  49: person
  50: pizza
  51: potted plant
  52: refrigerator
  53: remote
  54: sandwich
  55: scissors
  56: sheep
  57: sink
  58: skateboard
  59: skis
  60: snowboard
  61: spoon
  62:

## COCO Eval Sanity Check


In [11]:

num_eval_images = len(list(COCO_EVAL_IMAGES_DIR.glob("*.jpg")))
num_eval_labels = len(list(COCO_EVAL_LABELS_DIR.glob("*.txt")))

non_empty_labels = 0
total_label_lines = 0

for label_file in COCO_EVAL_LABELS_DIR.glob("*.txt"):
    text = label_file.read_text().strip()
    if text:
        non_empty_labels += 1
        total_label_lines += len(text.splitlines())

print("COCO eval root:", COCO_EVAL_ROOT)
print("Images folder:", COCO_EVAL_IMAGES_DIR)
print("Labels folder:", COCO_EVAL_LABELS_DIR)
print("Images:", num_eval_images)
print("Label files:", num_eval_labels)
print("Non-empty label files:", non_empty_labels)
print("Total label lines / instances:", total_label_lines)

assert num_eval_images == 5000, f"Expected 5000 COCO val images, found {num_eval_images}"
assert num_eval_labels == 5000, f"Expected 5000 COCO label files, found {num_eval_labels}"
assert total_label_lines > 0, "Total label lines is 0. Evaluation would be invalid."
assert COCO_EVAL_YAML.exists(), f"Missing YAML: {COCO_EVAL_YAML}"


COCO eval root: /root/coco_val2017_yolo_REAL
Images folder: /root/coco_val2017_yolo_REAL/images/val2017
Labels folder: /root/coco_val2017_yolo_REAL/labels/val2017
Images: 5000
Label files: 5000
Non-empty label files: 4952
Total label lines / instances: 36334


## Evaluate On COCO Val2017


In [12]:

trained_model = YOLO(str(TRAINED_WEIGHTS))

coco_ultra_metrics = val_with_batch_fallback(
    trained_model,
    data=COCO_EVAL_YAML,
    name=f"{RUN_NAME}_REAL_coco_val2017_eval_fixed_labels",
)

print("\n==============================")
print("REAL COCO val2017 Ultralytics results")
print("==============================")
print("mAP50-95:", coco_ultra_metrics.box.map)
print("mAP50:", coco_ultra_metrics.box.map50)
print("mAP75:", coco_ultra_metrics.box.map75)
print("Results saved to:", coco_ultra_metrics.save_dir)



Starting validation with batch=64
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu130 CUDA:0 (NVIDIA RTX PRO 4000 Blackwell, 23988MiB)
YOLO26x summary (fused): 190 layers, 55,727,103 parameters, 0 gradients, 193.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 446.0±180.3 MB/s, size: 189.1 KB)
val: Scanning /root/coco_val2017_yolo_REAL/labels/val2017... 5000 images, 48 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000 994.9it/s 5.0s0.0s
val: New cache created: /root/coco_val2017_yolo_REAL/labels/val2017.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 79/79 1.5s/it 1:560.5sss
                   all       5000      36334      0.646      0.509      0.546      0.394
              airplane         97        143      0.774      0.734      0.801      0.619
                 apple         76        236      0.546      0.305      0.302      0.204
              backpack        228        371      0.397      0.218   